# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("ML: ALS", 
                "spark://spark-master:7077")
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 02:39:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Songs recommednation

In [2]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = su.spark.createDataFrame(data, schema)
interactions_df.show()

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [3]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5
Number of users (m):3


In [4]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [5]:
model = als.fit(interactions_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [10]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=5)

# Show recommendations
user_recommendations.show(truncate=False)

[Stage 287:===================================================>  (95 + 2) / 100]

+-------+-------------------------------------------------------------------------------+
|user_id|recommendations                                                                |
+-------+-------------------------------------------------------------------------------+
|1      |[{2, 4.9621835}, {5, 4.8524218}, {1, 3.94023}, {3, 2.6195388}, {4, 2.5802584}] |
|2      |[{3, 3.9447005}, {2, 2.9733193}, {4, 2.905047}, {1, 2.2751427}, {5, 1.8554894}]|
|3      |[{3, 4.83555}, {4, 3.3364456}, {2, 2.6269844}, {1, 1.9627422}, {5, 1.0426705}] |
+-------+-------------------------------------------------------------------------------+



In [11]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])
songs_df = su.spark.createDataFrame(songs, songs_schema)

In [12]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song b|4.9621835|
|1      |song e|4.8524218|
|1      |song a|3.94023  |
|1      |song c|2.6195388|
|1      |song d|2.5802584|
|2      |song c|3.9447005|
|2      |song b|2.9733193|
|2      |song d|2.905047 |
|2      |song a|2.2751427|
|2      |song e|1.8554894|
|3      |song c|4.83555  |
|3      |song d|3.3364456|
|3      |song b|2.6269844|
|3      |song a|1.9627422|
|3      |song e|1.0426705|
+-------+------+---------+



In [13]:
predictions = model.transform(interactions_df)
predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.94023   |
|1      |2      |5     |4.9621835 |
|1      |5      |5     |4.8524218 |
|2      |2      |3     |2.9733193 |
|3      |1      |2     |1.9627422 |
|3      |3      |5     |4.83555   |
|3      |5      |1     |1.0426705 |
|2      |3      |4     |3.9447005 |
|2      |4      |3     |2.905047  |
+-------+-------+------+----------+



In [14]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.08807865441353031


# Lab 12: Building a Recommendation System with ALS 

In [15]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = su.spark.read \
                    .option("header", "false") \
                    .option("delimiter", "::") \
                    .schema(movies_ratings_schema) \
                    .csv(movies_ratings_path)

movies_ratings_df.printSchema()
movies_ratings_df.show(n=3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


## Create & Train the ML Model

## Persist the model

## Predictions

## Test ML Model

In [ ]:
su.spark.stop()